# 练习 1：async / await 协程并发

**AI 应用场景**

- 同时调用多个 LLM（比如 3 路模型投票）
- 同时搜索多个数据源（天气 + 新闻 + 数据库）
- 不用串行等待，总耗时 ≈ 最慢那一次

**核心语法**

| 写法 | 作用 |
|------|------|
| `async def` | 声明协程函数（定义时不会立刻执行） |
| `await` | 挂起等待，但**不阻塞**其他协程 |
| `asyncio.gather()` | 把多个协程一起扔出去并发执行 |

> 📓 **Notebook 特别说明**：Jupyter 自带事件循环，**不需要 `asyncio.run()`**，直接在格子顶层写 `await` 就能跑（普通 .py 文件才需要用 `asyncio.run()` 包一层）。

In [1]:
import asyncio
import time

async def call_api(name: str, delay: float) -> str:
    # 模拟一次 API 调用，耗时 delay 秒
    print(f"  {name} 开始调用...")
    await asyncio.sleep(delay)  # 模拟网络等待（挂起，不占 CPU）
    return f"{name} 的结果"

print("call_api 已定义 ✅")

call_api 已定义 ✅


## 串行执行：一次等一个

`await` 写三次、顺序执行 → 总耗时 = 3 × 0.5s

In [2]:
start = time.time()

await call_api("串行-1", 0.5)
await call_api("串行-2", 0.5)
await call_api("串行-3", 0.5)

print(f"→ 串行总耗时: {time.time() - start:.2f}s")

  串行-1 开始调用...


  串行-2 开始调用...


  串行-3 开始调用...


→ 串行总耗时: 1.50s


## 并发执行：三个同时发出去

`asyncio.gather()` 把 3 个协程**一起**交给事件循环 → 总耗时 ≈ 最慢那一个

In [3]:
start = time.time()

results = await asyncio.gather(
    call_api("并发-1", 0.5),
    call_api("并发-2", 0.5),
    call_api("并发-3", 0.5),
)

print(f"→ 并发总耗时: {time.time() - start:.2f}s")
print(f"→ 返回结果: {results}")

  并发-1 开始调用...
  并发-2 开始调用...
  并发-3 开始调用...


→ 并发总耗时: 0.50s
→ 返回结果: ['并发-1 的结果', '并发-2 的结果', '并发-3 的结果']


## 💡 小结

- 串行 1.5s，并发 0.5s —— **协程不是同时算，而是"等的时候去干别的"**
- `asyncio.sleep()` 是假等待（挂起）；真实场景换成 `await client.chat.completions.create(...)` 就是真·并发调用 LLM
- 你项目里 `common/llm.py` 目前是同步的；以后做"多路模型投票"就会用到这里的技术